In [ ]:
import joblib # Added import for joblib
model = joblib.load("/content/drive/MyDrive/פרויקט גמר/xgb_model (1).pkl")


In [ ]:
import gradio as gr
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib # Added import for joblib

def predict_logic(symbol):
    try:
        # 1. הורדת הנתונים
        df = yf.download(symbol, period="1mo", interval="1d")

        # 2. התיקון הקריטי: ביטול הכותרות הכפולות (Multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 3. מחיקת העמודה רק אם היא קיימת (זה ימנע את השגיאה שראית בתמונה)
        if 'Adj Close' in df.columns:
            df = df.drop(columns=['Adj Close'])

        # 4. המשך הלוגיקה של המודל שלך...
        # למשל: features = prepare_features(df)
        # prediction = model.predict(features)

        return "הצלחה!", "כאן יבוא החיזוי" # דוגמה להחזרה

    except Exception as e:
        # זה מה שמציג את השגיאה שראית בתמונה
        return f"Error: {str(e)}", None

# 1. טעינת המודל (החלף את השם לשם הקובץ שלך)
model = joblib.load("/content/drive/MyDrive/פרויקט גמר/xgb_model (1).pkl")
# Removed: model = xgb_model(1).pkl() (Incorrect syntax)
# Removed: model.load_model("my_xgboost_model.json") (Conflicting with joblib load)

def get_features(symbol):
    # משיכת נתונים מספיקים ליצירת Features (למשל חודש אחרון)
  #  symbol = "^GSPC"
    df = yf.download(symbol, period="1mo", interval="1d", multi_level_index=False)
    print("GOOD morning")
    # df = yf.download(symbol, period="1mo", interval="1d")
    if df.empty: return None

    # יצירת ה-Features בדיוק כפי שעשית במחברת (דוגמאות נפוצות):
    df['Returns'] = df['Close'].pct_change()
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['Volatility'] = df['Returns'].rolling(window=5).std()

    # לקיחת השורה האחרונה ביותר לחיזוי
    latest_features = df.tail(1).drop(columns=['Adj Close']) # התאם לעמודות שלך
    return latest_features, df

def predict_logic(symbol):
    try:
        features_row, full_df = get_features(symbol)

        # הרצת החיזוי (כאן המודל שלך נכנס לפעולה)
        # prediction = model.predict(features_row)[0] # 0 או 1
        # prob = model.predict_proba(features_row)[0] # הסתברות

        # --- דוגמה ויזואלית ללא המודל הטעון ---
        last_price = full_df['Close'].iloc[-1]
        prev_price = full_df['Close'].iloc[-2]
        trend = "UP 🚀" if last_price > prev_price else "DOWN 📉"
        conf = {"עליה": 0.65, "ירידה": 0.35} # כאן יבוא ה-prob מהמודל

        # יצירת גרף היסטורי קטן
        fig, ax = plt.subplots(figsize=(6, 3))
        full_df['Close'].tail(15).plot(ax=ax, color='blue', lw=2)
        ax.set_title(f"Price History: {symbol}")
        plt.tight_layout()

        return trend, conf, fig

    except Exception as e:
        return f"Error: {str(e)}", None, None

# 2. בניית ממשק המשתמש (UI)
with gr.Blocks(title="S&P 500 Predictor") as demo:
    gr.Markdown("# 🤖 חוזה מגמת מניות - AI Project")
    gr.Markdown("האפליקציה מושכת נתונים מ-Yahoo Finance ומריצה מודל XGBoost לחיזוי המגמה.")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל ^GSPC או AAPL)", value="^GSPC")
        btn = gr.Button("נתח עכשיו", variant="primary")

    with gr.Row():
        with gr.Column():
            out_label = gr.Textbox(label="תחזית המודל (עליה/ירידה)")
            out_prob = gr.Label(label="הסתברות")
        with gr.Column():
            out_plot = gr.Plot(label="גרף מחירים אחרון")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[out_label, out_prob, out_plot])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d60faeabb5579f856.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import yfinance as yf

symbol = "^GSPC"
df = yf.download(symbol, period="1mo", interval="1d", multi_level_index=False)
print(df.head(10))

/tmp/ipykernel_445/644755225.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period="1mo", interval="1d", multi_level_index=False)
[*********************100%***********************]  1 of 1 completed

                  Close         High          Low         Open      Volume
Date                                                                      
2026-02-17  6843.220215  6866.990234  6775.500000  6819.859863  5418480000
2026-02-18  6881.310059  6909.120117  6849.660156  6855.479980  5098160000
2026-02-19  6861.890137  6879.120117  6833.060059  6861.339844  5151690000
2026-02-20  6909.509766  6915.859863  6836.330078  6843.259766  5432480000
2026-02-23  6837.750000  6916.959961  6819.819824  6901.250000  5638350000
2026-02-24  6890.069824  6899.169922  6815.430176  6837.370117  5266090000
2026-02-25  6946.129883  6952.509766  6915.149902  6915.149902  5328060000
2026-02-26  6908.859863  6947.250000  6859.729980  6944.740234  5889550000
2026-02-27  6878.879883  6882.959961  6831.740234  6856.540039  6665660000
2026-03-02  6881.620117  6901.009766  6796.850098  6824.359863  6079080000


In [ ]:
import gradio as gr
import yfinance as yf
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib # Ensure joblib is imported for model loading

# --- 1. טעינת המודל ---
# This model loading is crucial for the predict_logic function
# It should be at the global scope, not commented out.
model = joblib.load("/content/drive/MyDrive/פרויקט גמר/xgb_model (1).pkl")

def predict_logic(symbol):
    try:
        # א. הורדת נתונים
        df = yf.download(symbol, period="1mo", interval="1d")

        if df.empty:
            return "לא נמצאו נתונים עבור הסימול הזה", None, None

        # ב. התיקון הקריטי: "ניקוי" כותרות כפולות (Multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            if 'Ticker' in df.columns:
                df = df.drop(columns=['Ticker'])

        # ג. מחיקת עמודות בשיטה בטוחה
        available_cols = df.columns.tolist()
        needed_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        final_df = df[[col for col in needed_cols if col in available_cols]].copy()

        # ד. לוגיקת החיזוי - Feature Engineering
        if 'Close' in final_df.columns:
            final_df['Returns'] = final_df['Close'].pct_change()
            final_df['MA5'] = final_df['Close'].rolling(window=5).mean()
            final_df['MA20'] = final_df['Close'].rolling(window=20).mean()
            final_df['Volatility'] = final_df['Returns'].rolling(window=5).std()
            final_df = final_df.dropna()

        if final_df.empty:
            return "לא ניתן לחשב מאפיינים", None, None

        # Prepare X_input for the model
        feature_columns = ['Returns', 'MA5', 'MA20', 'Volatility'] # Adjust as per your model's training features
        X_input_columns = [col for col in feature_columns if col in final_df.columns]

        if not X_input_columns:
            return "לא נמצאו מספיק מאפיינים לחיזוי", None, None

        X_input = final_df[X_input_columns].tail(1)

        if X_input.empty:
            return "לא נמצאו נתונים מספיקים ליצירת features", None, None

        # Model prediction logic
        # Assuming 'model' is loaded and has the 'predict_proba' method
        probs = model.predict_proba(X_input)[0]
        confidence_up = probs[1]   # Assuming class 1 corresponds to 'up'
        confidence_down = probs[0] # Assuming class 0 corresponds to 'down'

        result_text = "מגמת עלייה 🚀" if confidence_up > confidence_down else "מגמת ירידה 📉"
        result_probs = {"עלייה": float(confidence_up), "ירידה": float(confidence_down)}

        # יצירת גרף
        fig, ax = plt.subplots(figsize=(6, 3))
        plot_color = 'green' if confidence_up > confidence_down else 'red'
        final_df['Close'].tail(15).plot(ax=ax, color=plot_color, lw=2)
        ax.set_title(f"Price: {symbol}")
        plt.tight_layout()

        return result_text, result_probs, fig

    except Exception as e:
        # אם יש שגיאה, נציג אותה בצורה ברורה
        return f"שגיאה בתהליך: {str(e)}", None, None

# --- 2. ממשק ה-Gradio ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 חוזה מגמת S&P 500")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל ^GSPC)", value="^GSPC")
        btn = gr.Button("בצע חיזוי", variant="primary")

    with gr.Row():
        with gr.Column():
            res_text = gr.Textbox(label="תחזית")
            res_label = gr.Label(label="הסתברות")
        with gr.Column():
            res_plot = gr.Plot(label="גרף מחירים אחרון")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[res_text, res_label, res_plot])

demo.launch()

/usr/lib/python3.12/pickle.py:1760: UserWarning: [11:18:34] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)
/tmp/ipykernel_215/2563226316.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8fd78060294b88ca96.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import yfinance as yf
import pandas as pd
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

# --- 1. טעינת המודל ---
try:
    model = joblib.load("my_xgboost_model.pkl")
    # שליפת שמות העמודות שהמודל צופה להן (חשוב מאוד!)
    if hasattr(model, "feature_names_in_"):
        expected_features = model.feature_names_in_.tolist()
    else:
        # אם זה מודל ישן יותר, ננסה להוציא מהבוסטר
        expected_features = model.get_booster().feature_names
    print(f"Model loaded. Expected features: {expected_features}")
except Exception as e:
    print(f"Error loading model: {e}")
    expected_features = []

def predict_logic(symbol):
    try:
        # א. הורדת נתונים - שימוש ב-multi_level_index=False למניעת כותרות כפולות
        df = yf.download(symbol, period="6mo", interval="1d", multi_level_index=False)

        if df.empty:
            return "לא נמצאו נתונים עבור הסימול הזה", None, None

        # ב. ניקוי ושיטוח כותרות (ליתר ביטחון)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # ג. יצירת אינדיקטורים טכניים (הוסף כאן את מה שהשתמשת בו במחברת)
        # דוגמאות נפוצות:
        df['MA7'] = df['Close'].rolling(window=7).mean()
        df['MA21'] = df['Close'].rolling(window=21).mean()
        df['Daily_Return'] = df['Close'].pct_change()

        # ד. טיפול בערכים חסרים (NaN) שנוצרו מהחישובים
        df = df.ffill().dropna()

        # ה. השלב הקריטי: סידור העמודות לפי דרישת המודל
        # אם עמודה חסרה (כמו Adj Close), הקוד יצור אותה עם אפסים כדי לא לקרוס
        X_input = df.reindex(columns=expected_features, fill_value=0)

        # לקיחת השורה האחרונה בלבד לחיזוי
        latest_row = X_input.tail(1)

        # ו. חיזוי הסתברות
        probabilities = model.predict_proba(latest_row)[0]
        prob_up = float(probabilities[1])
        prob_down = float(probabilities[0])

        # ז. תוצאות לממשק
        trend_text = "עליה (UP) 🚀" if prob_up > prob_down else "ירידה (DOWN) 📉"
        prob_results = {"עליה": prob_up, "ירידה": prob_down}

        # ח. יצירת גרף
        fig, ax = plt.subplots(figsize=(6, 3))
        df['Close'].tail(30).plot(ax=ax, color='blue', title=f"Price History: {symbol}")
        plt.tight_layout()

        return trend_text, prob_results, fig

    except Exception as e:
        return f"שגיאה בחיזוי: {str(e)}", None, None

# --- 2. ממשק Gradio ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📈 מערכת חיזוי מגמות AI")
    gr.Markdown("הזן סימול מ-Yahoo Finance (כמו ^GSPC, AAPL, GC=F) לקבלת הסתברות מהמודל.")

    with gr.Row():
        ticker_input = gr.Textbox(label="סימול נכס", value="^GSPC")
        submit_btn = gr.Button("נתח עכשיו", variant="primary")

    with gr.Row():
        with gr.Column():
            res_trend = gr.Textbox(label="תחזית")
            res_label = gr.Label(label="ביטחון המודל (Confidence)")
        with gr.Column():
            res_plot = gr.Plot(label="גרף מחיר אחרון")

    submit_btn.click(
        fn=predict_logic,
        inputs=ticker_input,
        outputs=[res_trend, res_label, res_plot]
    )

if __name__ == "__main__":
    demo.launch()

Error loading model: [Errno 2] No such file or directory: 'my_xgboost_model.pkl'


/tmp/ipykernel_215/2215004970.py:70: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38574fcd1407b94e3c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import yfinance as yf
import pandas as pd
import xgboost as xgb
import joblib

# --- טעינת המודל ---
model = joblib.load("/content/drive/MyDrive/פרויקט גמר/xgb_model (1).pkl")

# זיהוי אוטומטי של העמודות שהמודל צריך
if hasattr(model, "feature_names_in_"):
    expected_features = model.feature_names_in_.tolist()
else:
    expected_features = model.get_booster().feature_names

def predict_logic(symbol):
    try:
        # 1. הורדת נתונים (תקופה ארוכה כדי שלא יחסר כלום)
        df = yf.download(symbol, period="2y", interval="1d", multi_level_index=False)

        if df.empty:
            return "שגיאה: לא ירדו נתונים מ-Yahoo Finance", None

        # 2. ניקוי כותרות
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 3. יצירת אינדיקטורים (חשוב: תוודא שאלו השמות שהמודל מכיר)
        # אם המודל שלך השתמש ב-RSI או ממוצעים אחרים, צריך להוסיף אותם כאן
        df['MA7'] = df['Close'].rolling(window=7).mean()
        df['MA21'] = df['Close'].rolling(window=21).mean()

        # 4. התאמה מושלמת לעמודות המודל
        # הפעולה הזו יוצרת את העמודות החסרות עם 0 כדי שהקוד לא יקרוס
        X_input = df.reindex(columns=expected_features).ffill().bfill()

        # בדיקה אם נשארו ערכים ריקים בטבלה
        if X_input.isnull().values.any():
            missing = X_input.columns[X_input.isnull().any()].tolist()
            return f"שגיאה: חסרים נתונים עבור העמודות: {missing}", None

        # 5. חיזוי על השורה האחרונה
        latest_row = X_input.tail(1)

        # בדיקה שהשורה לא ריקה
        if latest_row.empty:
            return "שגיאה: השורה האחרונה לחיזוי ריקה", None

        # הרצת החיזוי
        probs = model.predict_proba(latest_row)[0]

        prob_up = float(probs[1])
        prob_down = float(probs[0])

        res_text = "עליה 🚀" if prob_up > prob_down else "ירידה 📉"
        res_probs = {"עליה": prob_up, "ירידה": prob_down}

        return res_text, res_probs

    except Exception as e:
        return f"שגיאה פנימית: {str(e)}", None

# --- ממשק פשוט לבדיקה ---
demo = gr.Interface(
    fn=predict_logic,
    inputs=gr.Textbox(label="סימול (למשל ^GSPC)"),
    outputs=[gr.Textbox(label="תוצאה"), gr.Label(label="הסתברות")],
    title="בודק מודל XGBoost"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7196e6b1757bf612c5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import subprocess
import time

# --- 1. פונקציית הלוגיקה של האפליקציה שלך ---
def greet(name):
    return f"שלום {name}, האפליקציה רצה דרך מנהרה מאובטחת!"

# --- 2. הגדרת הממשק של Gradio ---
demo = gr.Interface(fn=greet, inputs="text", outputs="text")

# --- 3. הפעלת המנהרה (SSH Tunnel) באופן אוטומטי ---
def start_tunnel(port):
    # הפקודה שמריצה את המנהרה דרך localhost.run
    # nokey@localhost.run מאפשר חיבור ללא הרשמה
    ssh_command = f"ssh -R 80:localhost:{port} nokey@localhost.run"

    print(f"\n[!] פותח מנהרה ציבורית לפורט {port}...")

    # מריץ את הפקודה בתהליך רקע
    process = subprocess.Popen(
        ssh_command,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    # מחכה רגע כדי לקרוא את הכתובת שנוצרה מהפלט
    time.sleep(2)
    for line in process.stdout:
        if ".lhr.life" in line:
            print(f"\n" + "="*40)
            print(f"✅ הכתובת הציבורית שלך היא: {line.strip()}")
            print("="*40 + "\n")
            break

# --- 4. הרצה של הכל ביחד ---
if __name__ == "__main__":
    PORT = 7860

    # הפעלת המנהרה
    start_tunnel(PORT)

    # הפעלת Gradio (ללא share=True כי המנהרה עושה את העבודה)
    demo.launch(server_port=PORT)


[!] פותח מנהרה ציבורית לפורט 7860...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7dcf79ca965c0b22e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import yfinance as yf
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import subprocess
import time
import sys
import threading # Import threading to run tasks concurrently

# --- Shared variable to hold the Gradio port ---
_gradio_port = [None] # Use a list to make it mutable within the thread

# --- פונקציה להפעלת המנהרה (SSH Tunnel) ---
def start_tunnel(port):
    # הפקודה שיוצרת חיבור לעולם דרך localhost.run
    ssh_command = f"ssh -o StrictHostKeyChecking=no -R 80:localhost:{port} nokey@localhost.run"

    print(f"\n[!] מנסה לפתוח מנהרה ציבורית בפורט {port}...")

    # הפעלה כתהליך רקע
    process = subprocess.Popen(
        ssh_command,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # המתנה קלה וחיפוש הכתובת הציבורית בפלט
    time.sleep(3)
    found_url = False
    # Try reading multiple lines for the URL
    for _ in range(15): # Increased attempts
        line = process.stdout.readline()
        if line and (".lhr.life" in line or "localhost.run" in line):
            print("\n" + "="*50)
            print(f"🚀 האפליקציה שלך זמינה בכתובת: {line.strip()}")
            print("="*50 + "\n")
            found_url = True
            break
        time.sleep(0.5) # Wait a bit longer between reads
    if not found_url:
        print("[?] המנהרה הופעלה. אם לא ראית כתובת, בדוק אם ה-SSH מאושר במחשב.")

# --- פונקציה להפעלת Gradio ---
def run_gradio_app(demo_instance):
    try:
        # Launch Gradio, letting it find a free port.
        # server_port=0 ensures Gradio picks an available port.
        # share=False ensures we use our own tunnel, not Gradio's sharing.
        # Removed quiet=True to see potential startup errors
        launch_info = demo_instance.launch(server_port=0, share=False)
        _gradio_port[0] = launch_info.port # Store the actual port Gradio is using
        print(f"Gradio server successfully started on port: {launch_info.port}")
    except Exception as e:
        print(f"Error launching Gradio in thread: {e}", file=sys.stderr)
        import traceback
        traceback.print_exc(file=sys.stderr)
        _gradio_port[0] = -1 # Indicate failure

# --- 1. לוגיקת החיזוי ---
def predict_logic(symbol):
    try:
        # א. הורדת נתונים
        df = yf.download(symbol, period="1mo", interval="1d")

        if df.empty:
            return "לא נמצאו נתונים עבור הסימול הזה", None, None

        # ב. תיקון כותרות כפולות (Multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            if 'Ticker' in df.columns:
                df = df.drop(columns=['Ticker'])

        # ג. בחירת עמודות רלוונטיות
        available_cols = df.columns.tolist()
        needed_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        final_df = df[[col for col in needed_cols if col in available_cols]].copy()

        # ד. לוגיקת החיזוי (simplified for this example)
        last_price = final_df['Close'].iloc[-1]
        prev_price = final_df['Close'].iloc[-2]

        trend = "עליה (UP) 🚀" if last_price > prev_price else "ירידה (DOWN) 📉"
        prob = {"עליה": 0.7, "ירידה": 0.3} # Placeholder probabilities

        # יצירת גרף
        fig, ax = plt.subplots(figsize=(6, 3))
        final_df['Close'].tail(15).plot(ax=ax, color='green' if last_price > prev_price else 'red')
        ax.set_title(f"Price: {symbol}")
        plt.tight_layout()

        return trend, prob, fig

    except Exception as e:
        return f"שגיאה בתהליך: {str(e)}", None, None

# --- 2. ממשק ה-Gradio ---
with gr.Blocks() as demo: # Removed theme parameter here
    gr.Markdown("# 🤖 חוזה מגמת S&P 500 (דרך מנהרת SSH)")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל ^GSPC)", value="^GSPC")
        btn = gr.Button("בצע חיזוי", variant="primary")

    with gr.Row():
        with gr.Column():
            res_text = gr.Textbox(label="תחזית")
            res_label = gr.Label(label="הסתברות")
        with gr.Column():
            res_plot = gr.Plot(label="גרף מחירים אחרון")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[res_text, res_label, res_plot])

# --- 3. הרצה משולבת ---
if __name__ == "__main__":
    # 1. Start Gradio in a separate thread so it doesn't block the main thread
    # This allows us to get its port and then start the tunnel.
    gradio_thread = threading.Thread(target=run_gradio_app, args=(demo,))
    gradio_thread.start()

    # 2. Wait for Gradio to start and set the port
    print("Waiting for Gradio app to start and find a port...")
    while _gradio_port[0] is None:
        time.sleep(1) # Wait for 1 second intervals

    actual_gradio_port = _gradio_port[0]

    if actual_gradio_port == -1:
        print("Gradio app failed to start. Cannot create tunnel.", file=sys.stderr)
    else:
        print(f"Gradio app running on internal port: {actual_gradio_port}")

        # 3. Start the SSH tunnel using the actual port Gradio is running on
        start_tunnel(actual_gradio_port)

    # The main thread will now wait for the gradio_thread to finish, which it won't until manually stopped.
    # Or, the script can just exit after setting up everything.

/tmp/ipykernel_215/1866336243.py:95: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo: # Gradio app definition


Waiting for Gradio app to start and find a port...


Exception in thread Thread-25 (run_gradio_app):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection.py", line 101, in handle_request
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection.py", line 78, in handle_request
    stream = self._connect(request)
     

In [1]:
import gradio as gr
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import time
import threading

# --- 1. פונקציית המנהרה (רץ בנפרד כדי לא לתקוע את התוכנית) ---
def start_ssh_tunnel(port):
    try:
        # פקודת SSH שמתחברת ל-localhost.run
        # -o StrictHostKeyChecking=no מונע שאלות אבטחה מעצבנות בחיבור ראשון
        cmd = f"ssh -o StrictHostKeyChecking=no -R 80:localhost:{port} nokey@localhost.run"

        print(f"\n[Tunnel] מנסה ליצור כתובת ציבורית עבור פורט {port}...")

        # הרצה של הפקודה
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

        # קריאת הפלט כדי למצוא את הכתובת הציבורית
        for line in iter(proc.stdout.readline, ""):
            if "lhr.life" in line or "localhost.run" in line:
                print("\n" + "="*50)
                print(f"🔗 הכתובת הציבורית שלך מוכנה:")
                print(f"👉 {line.strip()}")
                print("="*50 + "\n")
                break
    except Exception as e:
        print(f"[Tunnel Error] לא הצלחתי לפתוח מנהרה: {e}")

# --- 2. לוגיקת החיזוי של המניות ---
def predict_logic(symbol):
    try:
        # הורדת נתונים (חודש אחרון)
        df = yf.download(symbol, period="1mo", interval="1d")

        if df.empty:
            return "לא נמצאו נתונים עבור הסימול הזה", None, None

        # ניקוי כותרות (Multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            if 'Ticker' in df.columns:
                df = df.drop(columns=['Ticker'])

        # בחירת נתונים אחרונים
        last_price = float(df['Close'].iloc[-1])
        prev_price = float(df['Close'].iloc[-2])

        # קביעת מגמה (לוגיקה בסיסית)
        diff = last_price - prev_price
        trend = f"עליה (UP) 🚀" if diff > 0 else f"ירידה (DOWN) 📉"
        change_pct = (diff / prev_price) * 100

        res_text = f"מגמה: {trend}\nמחיר אחרון: {last_price:.2f}\nשינוי יומי: {change_pct:.2f}%"

        # הסתברות "דמה" (כאן תוכל להטמיע את המודל שלך בהמשך)
        prob = {"עליה": 0.65 if diff > 0 else 0.35, "ירידה": 0.35 if diff > 0 else 0.65}

        # יצירת גרף
        fig, ax = plt.subplots(figsize=(8, 4))
        df['Close'].plot(ax=ax, color='blue', marker='o')
        ax.set_title(f"Price History: {symbol}")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        return res_text, prob, fig

    except Exception as e:
        return f"שגיאה: {str(e)}", None, None

# --- 3. בניית הממשק ב-Gradio ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 חוזה מגמות בורסה")
    gr.Markdown("האפליקציה הזו רצה על המחשב שלך ומופצת לעולם דרך מנהרת SSH מאובטחת.")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל AAPL, TSLA, ^GSPC)", value="^GSPC")
        btn = gr.Button("נתח עכשיו", variant="primary")

    with gr.Row():
        with gr.Column():
            res_text = gr.Textbox(label="תוצאה")
            res_label = gr.Label(label="תחזית מודל (הערכה)")
        with gr.Column():
            res_plot = gr.Plot(label="גרף מחירים")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[res_text, res_label, res_plot])

# --- 4. הרצה ---
if __name__ == "__main__":
    PORT = 7860

    # הפעלת המנהרה בשרשור נפרד כדי שלא תעצור את ה-Gradio
    threading.Thread(target=start_ssh_tunnel, args=(PORT,), daemon=True).start()

    # הפעלת Gradio (בלי share=True)
    demo.launch(server_port=PORT)

/tmp/ipykernel_287/2147991410.py:74: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:



[Tunnel] מנסה ליצור כתובת ציבורית עבור פורט 7860...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

🔗 הכתובת הציבורית שלך מוכנה:
👉 Warning: Permanently added 'localhost.run' (RSA) to the list of known hosts.

* Running on public URL: https://1fdf9e117933ff3ab5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
